# 01 – Data Collection
**Project:** Electoral Democracy in India (1947–2025)  
**Author:** Sagar Maindola  
**Purpose:** Load all raw CSV/Excel datasets into Pandas DataFrames, audit their structure, and save them into a unified SQLite database and the `RawData/` folder.

---
## Workflow position
`01_data_collection` → 02_database_design → 03_data_cleaning → 04_feature_engineering → 05_eda → 06_indices → 07_statistical_models → 08_ml_models → 09_network_analysis → 10_reporting

## 0. Imports & Project Setup

In [16]:
import os
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# ── Project folder structure ──────────────────────────────────────────────
BASE_DIR   = os.getcwd()
RAW_DIR    = os.path.join(BASE_DIR, 'RawData')
PROC_DIR   = os.path.join(BASE_DIR, 'ProcessedData')
REF_DIR    = os.path.join(BASE_DIR, 'ReferenceData')
DB_PATH    = os.path.join(BASE_DIR, 'electoral_india.db')

for d in [RAW_DIR, PROC_DIR, REF_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project directories created.")
print(f"Database will be stored at: {DB_PATH}")

Project directories created.
Database will be stored at: c:\Users\sagar\OneDrive\Desktop\Research Papers\Electoral Democracy in India (1947–2025)\Python Jupyter\electoral_india.db


## 1. Load Raw Datasets

| Variable | Expected Filename |
|---|---|
| `election_results_1951_2019` | `election_results_1951_2019.csv` |
| `loksabha_1962_2019` | `loksabha_1962_2019.csv` |
| `election_results_2024` | `election_results_2024.csv` |
| `ref_party_master` | `ref_party_master_1962_2021.csv` |
| `constituency_summary` | `election_constituency_summary.csv` |
| `literacy_1951_2011` | `literacy_1951_2011.csv` |
| `parliament_1951_2014` | `parliament_1951_2014.csv` |
| `state_sdp_2011_2023` | `state_sdp_2011_2023.csv` |
| `state_gdp_share_1960_2023` | `state_gdp_share_1960_2023.csv` |

In [17]:
# ── File registry: add/remove files here only ─────────────────────────────
FILE_REGISTRY = {
    'election_results_1951_2019' : 'election_results_1951_2019.csv',
    'loksabha_1962_2019'         : 'loksabha_1962_2019.csv',
    'election_results_2024'      : 'election_results_2024.csv',
    'ref_party_master'           : 'ref_party_master_1962_2021.csv',
    'constituency_summary'       : 'election_constituency_summary.csv',
    'literacy_1951_2011'         : 'literacy_1951_2011.csv',
    'parliament_1951_2014'       : 'parliament_1951_2014.csv',
    'state_sdp_2011_2023'        : 'state_sdp_2011_2023.csv',
    'state_gdp_share_1960_2023'  : 'state_gdp_share_1960_2023.csv',
}

raw_dfs = {}
missing_files = []

for key, filename in FILE_REGISTRY.items():
    filepath = os.path.join(RAW_DIR, filename)
    if os.path.exists(filepath):
        try:
            raw_dfs[key] = pd.read_csv(filepath, low_memory=False)
            print(f"✅  {key:40s} → {raw_dfs[key].shape}")
        except Exception as e:
            print(f"❌  {key}: Failed to load — {e}")
    else:
        missing_files.append(filename)
        print(f"⚠️   {key:40s} → FILE NOT FOUND ({filename})")

if missing_files:
    print(f"\n{len(missing_files)} file(s) missing. Place them in RawData/ and re-run.")

✅  election_results_1951_2019               → (89840, 15)
✅  loksabha_1962_2019                       → (8047, 12)
✅  election_results_2024                    → (8902, 11)
✅  ref_party_master                         → (10153, 21)
✅  constituency_summary                     → (21680, 8)
✅  literacy_1951_2011                       → (7, 4)
✅  parliament_1951_2014                     → (74930, 11)
✅  state_sdp_2011_2023                      → (34, 15)
✅  state_gdp_share_1960_2023                → (291, 6)


## 2. Data Audit — Shape, Dtypes, Missing Values

In [18]:
def audit_df(name, df):
    print(f"\n{'='*60}")
    print(f"  DATASET: {name}")
    print(f"{'='*60}")
    print(f"  Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Memory : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print(f"\n  Missing values:")
        for col, n in missing.items():
            pct = n / len(df) * 100
            print(f"    {col:<35} {n:>7,} ({pct:5.1f}%)")
    else:
        print("  No missing values.")
    print(f"\n  Columns & dtypes:")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<40} {str(dtype)}")
    return {
        'dataset': name, 'rows': df.shape[0], 'cols': df.shape[1],
        'missing_cols': len(missing), 'total_missing': int(missing.sum())
    }

audit_summary = []
for key, df in raw_dfs.items():
    summary = audit_df(key, df)
    audit_summary.append(summary)

audit_df_summary = pd.DataFrame(audit_summary)
print("\n\n" + "="*60)
print("  AUDIT SUMMARY")
print("="*60)
print(audit_df_summary.to_string(index=False))


  DATASET: election_results_1951_2019
  Shape  : 89,840 rows × 15 columns
  Memory : 54.29 MB

  Missing values:
    id                                       11 (  0.0%)
    gender                                4,521 (  5.0%)
    party                                    16 (  0.0%)
    age                                  60,370 ( 67.2%)
    category                             55,958 ( 62.3%)
    votes_received                            6 (  0.0%)
    votes_received_perc                      29 (  0.0%)

  Columns & dtypes:
    id                                       float64
    state                                    object
    constitution                             object
    election_year                            object
    candidate                                object
    gender                                   object
    party                                    object
    age                                      float64
    category                                 obj

## 3. State Name Audit — Extract unique state names per dataset

In [19]:
# ── Identify which column holds state names in each dataset ──────────────
STATE_COL_MAP = {
    'election_results_1951_2019' : 'state',
    'loksabha_1962_2019'         : 'state',
    'election_results_2024'      : 'State',
    'ref_party_master'           : 'StateName',
    'constituency_summary'       : 'StateUT Code',
    'parliament_1951_2014'       : 'state',     # adjust if column differs
    'state_sdp_2011_2023'        : 'State',     # adjust if column differs
    'state_gdp_share_1960_2023'  : 'State',     # adjust if column differs
}

all_state_names = set()

print("Unique state name variants found per dataset:\n")
for key, col in STATE_COL_MAP.items():
    if key in raw_dfs and col in raw_dfs[key].columns:
        unique_states = raw_dfs[key][col].dropna().unique()
        all_state_names.update(unique_states)
        print(f"  {key} [{col}] → {len(unique_states)} unique values")
        for s in sorted(unique_states):
            print(f"      {s}")
        print()

print(f"\nTotal unique state name strings found across all datasets: {len(all_state_names)}")

# Save audit
state_audit_df = pd.DataFrame(sorted(all_state_names), columns=['raw_state_name'])
state_audit_df.to_csv(os.path.join(REF_DIR, 'state_name_audit.csv'), index=False)
print("Saved: ReferenceData/state_name_audit.csv")

Unique state name variants found per dataset:

  election_results_1951_2019 [state] → 86 unique values
      ANDAMAN AND NICOBAR ISLANDS
      ANDHRA PRADESH
      ARUNACHAL PRADESH
      ASSAM
      Ajmer
      Andaman And Nicobar Islands
      Andhra Pradesh
      Arunachal Pradesh
      Assam
      BIHAR
      Bhopal
      Bihar
      Bombay
      CHANDIGARH
      CHATTISGARH
      Chandigarh
      Chattisgarh
      Coorg
      DADRA AND NAGAR HAVELI
      DAMAN AND DIU
      Dadra And Nagar Haveli
      Delhi
      GOA
      GOA DAMAN AND DIU
      GUJARAT
      Goa
      Gujarat
      HARYANA
      HIMACHAL PRADESH
      Haryana
      Himachal Pradesh
      Hyderabad
      JHARKHAND
      Jammu And Kashmir
      Jharkhand
      KARNATAKA
      KERALA
      Karnataka
      Kerala
      Kutch
      LAKSHADWEEP
      Laccadive, Minicoy And Amindivi Islands
      Lakshadweep
      MADHYA PRADESH
      MAHARASHTRA
      MANIPUR
      MEGHALAYA
      MIZORAM
      Madhya Bharat
      Ma

## 4. Load Reference / Crosswalk Files

In [20]:
# Load the master state crosswalk and name variants
# These files should be in your ReferenceData/ folder
# (Copy them from the electoral_democracy_india/ output folder)

state_crosswalk = pd.read_csv(os.path.join(REF_DIR, 'state_crosswalk.csv'))
state_variants  = pd.read_csv(os.path.join(REF_DIR, 'state_name_variants.csv'))

print(f"State crosswalk loaded:  {state_crosswalk.shape}")
print(f"State variants loaded:   {state_variants.shape}")

# Build a lookup dictionary: raw_name → canonical_name
STATE_NAME_MAP = dict(zip(state_variants['variant'], state_variants['canonical_name']))
print(f"\nState name mapping dictionary: {len(STATE_NAME_MAP)} entries")

State crosswalk loaded:  (42, 8)
State variants loaded:   (78, 4)

State name mapping dictionary: 78 entries


## 5. Apply State Name Normalization (Preview Only)

In [21]:
def normalize_state(raw_name, year=None):
    """
    Normalize a raw state name to its canonical form.
    Uses direct lookup first, then case-insensitive fallback.
    year parameter reserved for future time-aware bifurcation logic.
    """
    if pd.isna(raw_name):
        return None

    raw_str = str(raw_name).strip()

    # Direct lookup
    if raw_str in STATE_NAME_MAP:
        return STATE_NAME_MAP[raw_str]

    # Case-insensitive lookup
    for variant, canonical in STATE_NAME_MAP.items():
        if raw_str.lower() == variant.lower():
            return canonical

    # Partial match fallback (log for review)
    return raw_str  # Return as-is, will be flagged in cleaning step

# Preview on one dataset
if 'election_results_1951_2019' in raw_dfs:
    df_preview = raw_dfs['election_results_1951_2019'].copy()
    df_preview['state_canonical'] = df_preview['state'].apply(normalize_state)

    # Check unmapped states
    unmapped = df_preview[df_preview['state'] != df_preview['state_canonical']]
    still_raw = df_preview[df_preview['state'] == df_preview['state_canonical']]

    print("Normalization preview on election_results_1951_2019:")
    print(f"  Total rows           : {len(df_preview):,}")
    print(f"  Successfully mapped  : {len(unmapped):,}")
    print(f"  Needs manual review  : {len(still_raw):,}")
    print("\nSample of mappings:")
    print(df_preview[['state', 'state_canonical']].drop_duplicates().head(20).to_string(index=False))

Normalization preview on election_results_1951_2019:
  Total rows           : 89,840
  Successfully mapped  : 52,184
  Needs manual review  : 37,656

Sample of mappings:
            state  state_canonical
        Hyderabad        Hyderabad
    Uttar Pradesh    Uttar Pradesh
           Bombay      Maharashtra
            Ajmer            Ajmer
Travancore Cochin           Kerala
        Rajasthan        Rajasthan
           Punjab           Punjab
   Madhya Pradesh   Madhya Pradesh
           Madras       Tamil Nadu
            Assam            Assam
           Orissa           Odisha
           Mysore        Karnataka
      West Bengal      West Bengal
            Bihar            Bihar
 Himachal Pradesh Himachal Pradesh
  Vindhya Pradesh   Madhya Pradesh
            Coorg            Coorg
            Delhi            Delhi
       Saurashtra       Saurashtra
    Madhya Bharat   Madhya Pradesh


## 6. Save Raw Data to SQLite (raw schema)

In [22]:
conn = sqlite3.connect(DB_PATH)

# Raw table name mapping
RAW_TABLE_MAP = {
    'election_results_1951_2019' : 'raw_election_results_1951_2019',
    'loksabha_1962_2019'         : 'raw_loksabha_1962_2019',
    'election_results_2024'      : 'raw_election_results_2024',
    'ref_party_master'           : 'raw_ref_party_master',
    'constituency_summary'       : 'raw_constituency_summary',
    'literacy_1951_2011'         : 'raw_literacy_1951_2011',
    'parliament_1951_2014'       : 'raw_parliament_1951_2014',
    'state_sdp_2011_2023'        : 'raw_state_sdp',
    'state_gdp_share_1960_2023'  : 'raw_state_gdp_share',
}

for key, table_name in RAW_TABLE_MAP.items():
    if key in raw_dfs:
        raw_dfs[key].to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"  ✅  {table_name:<45} → {len(raw_dfs[key]):,} rows")

# Also save reference tables
state_crosswalk.to_sql('ref_state_crosswalk', conn, if_exists='replace', index=False)
state_variants.to_sql('ref_state_name_variants', conn, if_exists='replace', index=False)

conn.commit()
conn.close()
print(f"\nAll raw data saved to SQLite: {DB_PATH}")

  ✅  raw_election_results_1951_2019                → 89,840 rows
  ✅  raw_loksabha_1962_2019                        → 8,047 rows
  ✅  raw_election_results_2024                     → 8,902 rows
  ✅  raw_ref_party_master                          → 10,153 rows
  ✅  raw_constituency_summary                      → 21,680 rows
  ✅  raw_literacy_1951_2011                        → 7 rows
  ✅  raw_parliament_1951_2014                      → 74,930 rows
  ✅  raw_state_sdp                                 → 34 rows
  ✅  raw_state_gdp_share                           → 291 rows

All raw data saved to SQLite: c:\Users\sagar\OneDrive\Desktop\Research Papers\Electoral Democracy in India (1947–2025)\Python Jupyter\electoral_india.db


## 7. Collection Log — Save Audit Report

In [23]:
from datetime import datetime

log_path = os.path.join(PROC_DIR, '01_collection_log.txt')
with open(log_path, 'w') as f:
    f.write(f"Electoral Democracy in India — Data Collection Log\n")
    f.write(f"Run date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Database : {DB_PATH}\n\n")
    f.write(audit_df_summary.to_string(index=False))
    f.write(f"\n\nFiles loaded: {list(raw_dfs.keys())}\n")
    if missing_files:
        f.write(f"Missing files: {missing_files}\n")

print(f"Collection log saved: {log_path}")
print("\n✅  01_data_collection.ipynb COMPLETE")
print("    Next step → Run 02_database_design.ipynb")

Collection log saved: c:\Users\sagar\OneDrive\Desktop\Research Papers\Electoral Democracy in India (1947–2025)\Python Jupyter\ProcessedData\01_collection_log.txt

✅  01_data_collection.ipynb COMPLETE
    Next step → Run 02_database_design.ipynb


## Exporting all the files from SQlite Database to CSV

In [ ]:
import os
import sqlite3
import pandas as pd

# --- paths ---
BASE_DIR = os.getcwd()                     # or set your project root manually
DB_PATH = os.path.join(BASE_DIR, "electoral_india.db")
EXPORT_DIR = os.path.join(BASE_DIR, "ProcessedData")

os.makedirs(EXPORT_DIR, exist_ok=True)

# --- connect to sqlite ---
conn = sqlite3.connect(DB_PATH)

# --- tables to export ---
tables_to_export = {
    "raw_election_results_1951_2019": "raw_election_results_1951_2019.csv",
    "raw_loksabha_1962_2019": "raw_loksabha_1962_2019.csv",
    "raw_election_results_2024": "raw_election_results_2024.csv",
    "raw_ref_party_master": "raw_ref_party_master.csv",
    "raw_constituency_summary": "raw_constituency_summary.csv",
    "raw_literacy_1951_2011": "raw_literacy_1951_2011.csv",
    "raw_parliament_1951_2014": "raw_parliament_1951_2014.csv",
    "raw_state_sdp": "raw_state_sdp.csv",
    "raw_state_gdp_share": "raw_state_gdp_share.csv"
}

# --- export loop ---
export_log = []

for table_name, file_name in tables_to_export.items():
    query = f"SELECT * FROM {table_name}"
    df = pd.read_sql_query(query, conn)
    
    output_path = os.path.join(EXPORT_DIR, file_name)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    
    export_log.append({
        "table_name": table_name,
        "file_name": file_name,
        "rows": len(df),
        "columns": len(df.columns),
        "path": output_path
    })
    
    print(f"✅ Exported {table_name:<35} → {file_name} ({len(df):,} rows)")

conn.close()

# --- export summary log ---
export_log_df = pd.DataFrame(export_log)
log_path = os.path.join(EXPORT_DIR, "export_summary.csv")
export_log_df.to_csv(log_path, index=False, encoding="utf-8-sig")

print("\nAll tables exported successfully.")
print(f"CSV files saved in: {EXPORT_DIR}")
print(f"Export log saved as: {log_path}")

✅ Exported raw_election_results_1951_2019      → raw_election_results_1951_2019.csv (89,840 rows)
✅ Exported raw_loksabha_1962_2019              → raw_loksabha_1962_2019.csv (8,047 rows)
✅ Exported raw_election_results_2024           → raw_election_results_2024.csv (8,902 rows)
✅ Exported raw_ref_party_master                → raw_ref_party_master.csv (10,153 rows)
✅ Exported raw_constituency_summary            → raw_constituency_summary.csv (21,680 rows)
✅ Exported raw_literacy_1951_2011              → raw_literacy_1951_2011.csv (7 rows)
✅ Exported raw_parliament_1951_2014            → raw_parliament_1951_2014.csv (74,930 rows)
✅ Exported raw_state_sdp                       → raw_state_sdp.csv (34 rows)
✅ Exported raw_state_gdp_share                 → raw_state_gdp_share.csv (291 rows)

All tables exported successfully.
CSV files saved in: c:\Users\sagar\OneDrive\Desktop\Research Papers\Electoral Democracy in India (1947–2025)\Python Jupyter\ProcessedData
Export log saved as: c:\Users\

---

## Importing Extracted Raw Data

In [27]:
raw_constituency_summary = pd.read_csv('ProcessedData/raw_constituency_summary.csv')
raw_election_results_1951_2019 = pd.read_csv('ProcessedData/raw_election_results_1951_2019.csv')
raw_election_results_2024 = pd.read_csv('ProcessedData/raw_election_results_2024.csv')
raw_literacy_1951_2011 = pd.read_csv('ProcessedData/raw_literacy_1951_2011.csv')
raw_loksabha_1962_2019 = pd.read_csv('ProcessedData/raw_loksabha_1962_2019.csv')
raw_parliament_1951_2014 = pd.read_csv('ProcessedData/raw_parliament_1951_2014.csv')
raw_ref_party_master = pd.read_csv('ProcessedData/raw_ref_party_master.csv')
raw_state_gdp_share = pd.read_csv('ProcessedData/raw_state_gdp_share.csv')
raw_state_sdp = pd.read_csv('ProcessedData/raw_state_sdp.csv')